In [1]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")


Groq model initialized successfully.


In [2]:
fitness_goal = """
I am a beginner and want to improve my general fitness.
I can exercise for 30 minutes a day, five days a week.
I prefer walking and simple bodyweight exercises.
I have no gym equipment.
I want a gradual and manageable routine.
"""

print(fitness_goal)


I am a beginner and want to improve my general fitness.
I can exercise for 30 minutes a day, five days a week.
I prefer walking and simple bodyweight exercises.
I have no gym equipment.
I want a gradual and manageable routine.



In [3]:
class FitnessGoal(BaseModel):
    experience_level: str = Field(
        description="User's fitness experience level"
    )
    primary_goal: str = Field(
        description="Main fitness objective"
    )
    available_minutes_per_day: int = Field(
        description="Available exercise time per day"
    )
    days_per_week: int = Field(
        description="Number of exercise days available per week"
    )
    preferred_activities: list[str] = Field(
        description="Activities preferred by the user"
    )
    available_equipment: list[str] = Field(
        description="Available exercise equipment, or an empty list"
    )
    limitations: list[str] = Field(
        description="Explicitly stated limitations; do not invent any"
    )


goal_parser = PydanticOutputParser(pydantic_object=FitnessGoal)

print("Fitness goal schema created.")

Fitness goal schema created.


In [4]:
goal_prompt = PromptTemplate(
    template="""
Extract the user's fitness goal into the required structured format.

Use only information provided by the user.
Do not invent medical conditions, injuries, or physical limitations.
If no limitations are mentioned, use an empty list.
Represent unavailable equipment with an empty list.

User's fitness goal:
{fitness_goal}

{format_instructions}
""",
    input_variables=["fitness_goal"],
    partial_variables={
        "format_instructions": goal_parser.get_format_instructions()
    }
)

goal_chain = goal_prompt | llm | goal_parser

structured_goal = goal_chain.invoke({
    "fitness_goal": fitness_goal
})

print(structured_goal.model_dump_json(indent=2))

{
  "experience_level": "beginner",
  "primary_goal": "general fitness",
  "available_minutes_per_day": 30,
  "days_per_week": 5,
  "preferred_activities": [
    "walking",
    "bodyweight exercises"
  ],
  "available_equipment": [],
  "limitations": []
}


In [5]:
class WorkoutDay(BaseModel):
    day: str = Field(description="Day of the week")
    activities: list[str] = Field(
        description="Exercises planned for this day"
    )
    duration_minutes: int = Field(
        description="Total planned exercise duration in minutes"
    )


class WeeklyWorkoutPlan(BaseModel):
    weekly_plan: list[WorkoutDay] = Field(
        description="Workout schedule for the week"
    )
    rest_days: list[str] = Field(
        description="Days reserved for rest or recovery"
    )
    safety_notes: list[str] = Field(
        description="Practical, general exercise safety advice"
    )


plan_parser = PydanticOutputParser(
    pydantic_object=WeeklyWorkoutPlan
)

print("Weekly workout plan schema created.")

Weekly workout plan schema created.


In [6]:
plan_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a fitness planning assistant. Create a gradual, "
        "beginner-friendly general fitness plan using only the supplied "
        "structured goal. Respect the user's time, preferences, and "
        "equipment. Include recovery days, sensible warm-ups, and "
        "manageable exercise intensity. Do not make medical claims. "
        "If the user has a health concern or exercise causes pain, advise "
        "them to seek appropriate professional guidance. Return only "
        "the requested structured output."
    ),
    (
        "human",
        """
Create a one-week workout plan based on this structured fitness goal:

{structured_goal}

Requirements:
- Include exactly seven calendar days in weekly_plan.
- Include exercise days and rest or recovery days.
- Do not exceed the user's available daily exercise time.
- Use preferred activities and available equipment.
- Keep the routine suitable for the stated experience level.
- Follow the required output format.

{format_instructions}
"""
    )
])

plan_chain = plan_prompt | llm | plan_parser

weekly_plan = plan_chain.invoke({
    "structured_goal": structured_goal.model_dump_json(indent=2),
    "format_instructions": plan_parser.get_format_instructions()
})

print(weekly_plan.model_dump_json(indent=2))

{
  "weekly_plan": [
    {
      "day": "Monday",
      "activities": [
        "5 min brisk walk (warm‑up)",
        "20 min bodyweight circuit (e.g., 2 rounds of 10 squats, 8 push‑ups, 12 lunges per leg, 30‑sec plank)",
        "5 min gentle stretch (cool‑down)"
      ],
      "duration_minutes": 30
    },
    {
      "day": "Tuesday",
      "activities": [
        "30 min steady brisk walk"
      ],
      "duration_minutes": 30
    },
    {
      "day": "Wednesday",
      "activities": [
        "5 min marching in place (warm‑up)",
        "20 min bodyweight strength (e.g., 3 rounds of 10 squats, 8 push‑ups, 12 reverse lunges per leg, 30‑sec side plank each side)",
        "5 min full‑body stretch (cool‑down)"
      ],
      "duration_minutes": 30
    },
    {
      "day": "Thursday",
      "activities": [
        "Rest day – light activity such as casual walking or stretching optional"
      ],
      "duration_minutes": 0
    },
    {
      "day": "Friday",
      "activities": [
  

In [7]:
assert isinstance(structured_goal, FitnessGoal)
assert structured_goal.experience_level.strip()
assert structured_goal.available_minutes_per_day > 0
assert 1 <= structured_goal.days_per_week <= 7

assert isinstance(weekly_plan, WeeklyWorkoutPlan)
assert len(weekly_plan.weekly_plan) == 7

for day in weekly_plan.weekly_plan:
    assert day.day.strip()
    assert isinstance(day.activities, list)
    assert day.duration_minutes >= 0
    assert day.duration_minutes <= structured_goal.available_minutes_per_day

assert isinstance(weekly_plan.rest_days, list)
assert isinstance(weekly_plan.safety_notes, list)

print("Validation passed.")
print("Fitness goal extracted successfully.")
print("Weekly workout plan generated successfully.")

Validation passed.
Fitness goal extracted successfully.
Weekly workout plan generated successfully.


In [8]:
for day in weekly_plan.weekly_plan:
    print(f"\n{day.day}")
    print("-" * 30)

    for activity in day.activities:
        print(f"- {activity}")

    print(f"Planned duration: {day.duration_minutes} minutes")

print("\nRest Days:")
for day in weekly_plan.rest_days:
    print(f"- {day}")

print("\nSafety Notes:")
for note in weekly_plan.safety_notes:
    print(f"- {note}")


Monday
------------------------------
- 5 min brisk walk (warm‑up)
- 20 min bodyweight circuit (e.g., 2 rounds of 10 squats, 8 push‑ups, 12 lunges per leg, 30‑sec plank)
- 5 min gentle stretch (cool‑down)
Planned duration: 30 minutes

Tuesday
------------------------------
- 30 min steady brisk walk
Planned duration: 30 minutes

Wednesday
------------------------------
- 5 min marching in place (warm‑up)
- 20 min bodyweight strength (e.g., 3 rounds of 10 squats, 8 push‑ups, 12 reverse lunges per leg, 30‑sec side plank each side)
- 5 min full‑body stretch (cool‑down)
Planned duration: 30 minutes

Thursday
------------------------------
- Rest day – light activity such as casual walking or stretching optional
Planned duration: 0 minutes

Friday
------------------------------
- 5 min easy walk (warm‑up)
- 20 min interval walk (1 min fast pace, 2 min moderate pace, repeat)
- 5 min cool‑down walk and stretch
Planned duration: 30 minutes

Saturday
------------------------------
- 5 min dyna

In [9]:
print("Lab 5 Exercise 6 completed successfully.")
print("The fitness goal was converted into structured data.")
print("A weekly workout plan was generated and validated.")

Lab 5 Exercise 6 completed successfully.
The fitness goal was converted into structured data.
A weekly workout plan was generated and validated.
